# Imports

##### standard python libraries

In [44]:
import os
from sqlalchemy import create_engine
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [45]:
from pysdl import SDLClient

In [46]:
import warnings
warnings.filterwarnings('ignore')

##### Custom Libraries

In [47]:
import psql_utilities as psql_utl
import sdl_utilities as sdl_utl
import flow_depth_utilities as hc_utl
import Library_sampling as sampling 

### Set Paths

In [48]:
### path to configuration file of database
path_conf_file = './../../config/database.ini'
### path where to download the Experiment Data from SDL
path_experiment_dataset = '/data/users/adistefano/ml_emulator/download_data'

### Database Connection

In [49]:
db_config = psql_utl.load_config(path_conf_file)
#db_conn = psql_utl.connect(db_config)
### connect to the database - sqlalchemy
engine = create_engine(db_config['db_uri'])

### Connect to SDL

In [50]:
client = SDLClient()
client.auth.login()

{'response': 200, 'message': 'Authenticated on SDL successfully.'}


{'message': 'User already attached as andrea.distefano@ingv.it.'}

## Probabilistic Tsunami Risk Analysis (PTRA)

Jupyter Notebook to calculate Probabilistic Tsunami Risk Assessment (PTRA) through hazard–vulnerability convolution, where tsunami hazard curves are convolved with vulnerability functions to estimate conditional loss exceedance probabilities and Loss Exceedance Curves (LECs), while accounting for uncertainty in both hazard and vulnerability.


In order to perform the analysis the user can select for a given value of *manning coefficient*:

  1. the value of *manning coefficient*
  2. the target site: Catania or Siracusa

Then we select from metadata the associated simulation run parameters, see *'./query_sampling_v1.sql'*. 

In order to obtain a nice plots we also extract the triangles of mesh associated to the PS sceanarios, see *'./query_sampling_v1.sql'*.

### setup

##### read query from file

In [51]:
query_bathy_files = open("./query_bathymetry_site.sql", "r")
query_bathy_files = query_bathy_files.read()

In [52]:
query_hazard_curves = open("./query_hazard_curves.sql", "r")
query_hazard_curves = query_hazard_curves.read()

##### set input variables

In [53]:
# manning coefficient
manning_coef = 0.03
# location of interest (CT,SR)
site = "CT"
# output results of sis analysis
RUNDIR = "./data/CT_0.03_1000"

### query result

##### extract bathymetry grid for a given site

In [54]:
dict_params = {'site_loc': site}

In [55]:
df_bathy_files = pd.read_sql(query_bathy_files, con=engine, params=dict_params)
df_bathy_files

DatabaseError: Execution failed on sql '---------------------------------------------------------------------------------------------------------------------------------
-- QUERY: Select bathymetry grid file for a given site
---------------------------------------------------------------------------------------------------------------------------------


-- select bathymetry grid files
select 
	split_part(ds.name, '.', 1) as site,
	ds.path as path_bathy_file
from datasets as ds
where
   ds.fk_exp_id='exp_ct_sr'
and
	ds.type = 'global_input'
and
	ds.name = CONCAT('C_','%(site_loc)s', '.grd')': (psycopg2.errors.SyntaxError) syntax error at or near "CT"
LINE 16:  ds.name = CONCAT('C_',''CT'', '.grd')
                                  ^

[SQL: ---------------------------------------------------------------------------------------------------------------------------------
-- QUERY: Select bathymetry grid file for a given site
-----------------------------------------------------------------

##### extract simualtion outputs for a given site and manning coefficient

In [56]:
dict_params = {'manning_coef': manning_coef,
               'site_loc': site}

In [57]:
query_hazard_curves_completed = query_hazard_curves % dict_params
df_scen = pl.read_database_uri(query_hazard_curves_completed,engine='connectorx',uri=db_config['db_uri'],partition_num=6)
df_scen

sr_id,id_scen,seismicity_type,magnitude,lon,lat,rates,path_nc_file
str,str,str,f64,f64,f64,list[f64],str
"""sr_60645""","""E01446N4259E02038N3783-BS-M809…","""BS""",8.09,19.8065,38.5016,"[0.0, 7.0323e-14, … 2.2812e-18]","""CT-SR_M100/BS_003/E01446N4259E…"
"""sr_60646""","""E01522N3585E03578N2685-BS-M809…","""BS""",8.09,21.654,33.1056,"[0.0, 7.5325e-10, … 4.5968e-11]","""CT-SR_M100/BS_003/E01522N3585E…"
"""sr_60647""","""E01267N3753E01646N3535-BS-M754…","""BS""",7.54,15.3075,37.3774,"[7.5817e-8, 2.4024e-8, … 3.1604e-8]","""CT-SR_M100/BS_003/E01267N3753E…"
"""sr_60648""","""E01522N3585E03578N2685-BS-M809…","""BS""",8.09,23.5328,33.1056,"[0.0, 5.7877e-10, … 2.9059e-10]","""CT-SR_M100/BS_003/E01522N3585E…"
"""sr_60649""","""E01470N3877E01631N3641-BS-M707…","""BS""",7.07,15.8733,37.3774,"[1.6038e-8, 2.1899e-8, … 5.0667e-8]","""CT-SR_M100/BS_003/E01470N3877E…"
…,…,…,…,…,…,…,…
"""sr_60976""","""E01470N3877E01631N3641-BS-M707…","""BS""",7.07,16.0399,36.7029,"[1.1131e-8, 2.4841e-8, … 1.1095e-8]","""CT-SR_M100/BS_003/E01470N3877E…"
"""sr_60977""","""E01267N3753E01646N3535-BS-M754…","""BS""",7.54,15.2345,36.9278,"[4.8320e-8, 4.2425e-10, … 6.3302e-9]","""CT-SR_M100/BS_003/E01267N3753E…"
"""sr_60978""","""E01446N4259E02038N3783-BS-M754…","""BS""",7.54,19.232,38.5016,"[0.0, 1.1364e-8, … 9.2442e-10]","""CT-SR_M100/BS_003/E01446N4259E…"


### Probabilistic Tsunami Risk Analysis

We compute inundation Hazard Curves based on flow depth for scenarios sample with Importance Sampling

##### Load results from Importance Sampling

In [22]:
### output dir
sampled_scenarios = f'{RUNDIR}/selected_scenarios.parquet'

In [23]:
### load sampled scenarios from importance sampling
df_samp = pl.read_parquet(sampled_scenarios)
### associate simulations output
df_samp_scen = df_samp.select('sr_id','new_rates_sis')\
                      .join(df_scen,how='inner',on='sr_id')
df_samp_scen = df_samp_scen.to_pandas()
df_samp_scen

,sr_id,new_rates_sis,id_scen,seismicity_type,magnitude,lon,lat,rates,path_nc_file
0,sr_51558,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",E01267N3753E01646N3535-BS-M600_E01530N3737_D01...,BS,6.00,15.3075,37.3774,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",CT-SR_M100/BS_003/E01267N3753E01646N3535-BS-M6...
1,sr_52666,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",E01267N3753E01646N3535-BS-M600_E01530N3737_D01...,BS,6.00,15.3075,37.3774,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",CT-SR_M100/BS_003/E01267N3753E01646N3535-BS-M6...
2,sr_51558,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",E01267N3753E01646N3535-BS-M600_E01530N3737_D01...,BS,6.00,15.3075,37.3774,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",CT-SR_M100/BS_003/E01267N3753E01646N3535-BS-M6...
3,sr_51558,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",E01267N3753E01646N3535-BS-M600_E01530N3737_D01...,BS,6.00,15.3075,37.3774,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",CT-SR_M100/BS_003/E01267N3753E01646N3535-BS-M6...
4,sr_51558,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",E01267N3753E01646N3535-BS-M600_E01530N3737_D01...,BS,6.00,15.3075,37.3774,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",CT-SR_M100/BS_003/E01267N3753E01646N3535-BS-M6...
...,...,...,...,...,...,...,...,...,...
998,sr_298984,"[0.0, 0.0, 0.000003, 0.0, 0.0, 0.0, 0.0, 0.000...",E02020N3739E02658N3366-PS-Mur_PYes_Hom-M902_E0...,PS,9.02,24.0900,34.6700,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",CT-SR_M100/PS_003/E02020N3739E02658N3366-PS-Mu...
999,sr_297419,"[0.0, 0.0, 0.000003, 0.0, 0.0, 0.0, 0.0, 0.000...",E02020N3739E02658N3366-PS-Mur_PYes_Var-M902_E0...,PS,9.02,23.1300,35.1000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",CT-SR_M100/PS_003/E02020N3739E02658N3366-PS-Mu...
1000,sr_310200,"[0.0, 0.0, 0.000004, 0.0, 0.0, 0.0, 0.0, 0.000...",E02020N3739E02658N3366-PS-Str_PYes_Var-M902_E0...,PS,9.02,23.6800,34.6100,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",CT-SR_M100/PS_003/E02020N3739E02658N3366-PS-St...
1001,sr_305797,"[0.0, 0.0, 0.000002, 0.0, 0.0, 0.0, 0.0, 0.000...",E02020N3739E02658N3366-PS-Mur_PYes_Hom-M902_E0...,PS,9.02,23.6800,34.6100,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",CT-SR_M100/PS_003/E02020N3739E02658N3366-PS-Mu...


##### download selected output files & bathymetry grid for the given site from SDL

In [24]:
### download bathymetry grid at the site
grid_file = df_bathy_files['path_bathy_file'][0]
sdl_utl.download_sdl_file(download_dir=path_experiment_dataset,file_sdl_path=grid_file,sdl_client=client)

......File: /data/users/adistefano/ml_emulator/download_data/CT-SR_M100/SIMULATION_SETUP/bati/C_CT.grd already exists


In [25]:
### download simulation output data
for i, (scenario, file_ct) in enumerate(zip(df_samp_scen['id_scen'], df_samp_scen['path_nc_file'])):
    print(f"{i}: Scenario {scenario}")
    ###
    sdl_utl.download_sdl_file(download_dir=path_experiment_dataset,file_sdl_path=file_ct,sdl_client=client)
    if i % 50 == 0:
        ### refresh token
        client.auth.login()
sdl_utl.check_download(df_paths=df_samp_scen, download_dir=path_experiment_dataset)

0: Scenario E01267N3753E01646N3535-BS-M600_E01530N3737_D010_S337D50R090_A000056_S006
......File: /data/users/adistefano/ml_emulator/download_data/CT-SR_M100/BS_003/E01267N3753E01646N3535-BS-M600_E01530N3737_D010_S337D50R090_A000056_S006/C_CT.nc already exists
{'response': 200, 'message': 'Authenticated on SDL successfully.'}
1: Scenario E01267N3753E01646N3535-BS-M600_E01530N3737_D010_S112D50R090_A000056_S006
......File: /data/users/adistefano/ml_emulator/download_data/CT-SR_M100/BS_003/E01267N3753E01646N3535-BS-M600_E01530N3737_D010_S112D50R090_A000056_S006/C_CT.nc already exists
2: Scenario E01267N3753E01646N3535-BS-M600_E01530N3737_D010_S337D50R090_A000056_S006
......File: /data/users/adistefano/ml_emulator/download_data/CT-SR_M100/BS_003/E01267N3753E01646N3535-BS-M600_E01530N3737_D010_S337D50R090_A000056_S006/C_CT.nc already exists
3: Scenario E01267N3753E01646N3535-BS-M600_E01530N3737_D010_S337D50R090_A000056_S006
......File: /data/users/adistefano/ml_emulator/download_data/CT-SR_M

'Donwload completed: all files have been successufully downloaded'

##### compute flow depth at a given poi

It takes about 1 minute for one thousand scenarios

In [60]:
ct_lat = 37.48
ct_lon = 15.08
sr_lat = 37.06
sr_lon = 15.29
# select point inland of interest
poi_lat = ct_lat
poi_lon = ct_lon

In [ ]:
#
### open grid file
bathy_grid_site =  hc_utl.get_simulation_results_from_file(ds_path=grid_file,path_exp=path_experiment_dataset)
# list of flow depth at given poi for all scenarios
list_flowdepth_site = []
for idx, row in df_samp_scen.iterrows():
    ### open file with simulation result
    sim_output = hc_utl.get_simulation_results_from_file(ds_path=row['path_nc_file'],path_exp=path_experiment_dataset)
    ### compute flow_depth
    flow_depth_ds = hc_utl.compute_flow_depth(sim_result=sim_output,bathy_grid=bathy_grid_site)
    ### select flow_depth at poi
    flow_depth_site = float(flow_depth_ds['z'].sel(lat=poi_lat, lon=poi_lon, method='nearest').values)
    ### append result
    list_flowdepth_site.append(flow_depth_site)

##### compute hazard curve at a given poi

In [54]:
thresholds = np.array([0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0, 1.5, 2.0, 2.5, 3.0, 3.6, 4.32, 5.18])

In [55]:
np_flowdepth_site = np.array(list_flowdepth_site)
rates = np.array(df_samp_scen['new_rates_sis'].tolist())
#
hazard_curves = sampling.exceedance_curve(thresholds, np_flowdepth_site, rates)
#
df_per_hazard_curve = pd.DataFrame({
    'thresholds': thresholds,
    'mean': np.mean(hazard_curves, axis=1),
    'p2': np.percentile(hazard_curves, 2, axis=1),
    'p16': np.percentile(hazard_curves, 16, axis=1),
    'p84': np.percentile(hazard_curves, 84, axis=1),
    'p98': np.percentile(hazard_curves, 98, axis=1),
})

##### plot hazard curve

In [ ]:
thresholds = df_per_hazard_curve["thresholds"]
mean_hc = df_per_hazard_curve["mean"]
p2_hc = df_per_hazard_curve["p2"]
p16_hc = df_per_hazard_curve["p16"]
p84_hc = df_per_hazard_curve["p84"]
p98_hc = df_per_hazard_curve["p98"]

fig, ax = plt.subplots(figsize=(12, 9))

# Curva media SIS
ax.plot(thresholds, mean_hc, color='tab:blue', linestyle='-.', linewidth=3, label='SIS Mean')

# Percentili interni (16° - 84°)
ax.plot(thresholds, p16_hc, color='tab:blue', linewidth=2, linestyle='--', label='16th - 84th perc.')
ax.plot(thresholds, p84_hc, color='tab:blue', linewidth=2, linestyle='--', label='')

# Percentili esterni (2° - 98°)
ax.plot(thresholds, p2_hc, color='tab:blue', linewidth=1.8, linestyle=':', label='2nd - 98th perc.')
ax.plot(thresholds, p98_hc, color='tab:blue', linewidth=1.8, linestyle=':', label='')

ax.set_yscale('log')
ax.set_ylim([1e-7, 1e-2])
ax.set_xlim([0, thresholds.max()])

ax.set_xlabel("Flow depth [m]", fontsize=18)
ax.set_ylabel("Exceedance-rate (1 year)", fontsize=18)
ax.set_title(f"Hazard Curve at {site}", fontsize=22)

ax.tick_params(axis='both', which='major', labelsize=16)
ax.legend(fontsize=16, frameon=False, loc="upper right")
ax.grid(True, which="both", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

##### save hazard curves data

In [ ]:
### save hazard curves for each rate
file_hazard_curves = f"1000_hazard_curves_{site}.txt"
path_hazard_curves = os.path.join(RUNDIR,file_hazard_curves)
print(f"Save hazard curves for each rate: {path_hazard_curves}")
np.savetxt(path_hazard_curves, hazard_curves, fmt="%.6e", delimiter=" ")

### save hazard curves stats
file_hazard_curves_perc = f"perc_hazard_curves_{site}.txt"
path_perc = os.path.join(RUNDIR,file_hazard_curves_perc)
print(f"Save hazard curve details: {path_hazard_curves}")
df_per_hazard_curve.to_csv(path_perc, sep=" ", index=False, float_format="%.6e")